# Práctica III: Aerolíneas y Satisfacción

Elena & Carlos Garrido del Toro

Inteligencia Artificial a 14 de Enero de 2026

---

## 1. Introducción y Objetivos

El objetivo principal de este ejercicio es aplicar conceptos y técnicas de **Aprendizaje Automático Supervisado** .Trabajaremos con el conjunto de datos *Airline Passenger Satisfaction Dataset*, el cual contiene información demográfica de pasajeros, detalles de sus vuelos y puntuaciones sobre servicios específicos (wifi, comida, comodidad, etc.).

El objetivo es predecir la variable `satisfaction` (si un pasajero está "Satisfecho" o "Neutral/Insatisfecho"). Para ello, compararemos el rendimiento de los siguientes modelos:

1.  Perceptrón
2.  Regresión Logística
3.  Máquinas de Vectores de Soporte (SVM)
4.  Árboles de Decisión
5.  Bosques Aleatorios (Random Forest)

In [7]:
# Importación de librerías fundamentales requeridas 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración para mostrar gráficos integrados
%matplotlib inline

print("Entorno configurado correctamente. Listo para cargar datos.")

Entorno configurado correctamente. Listo para cargar datos.


Importamos el DataSet descargado y ubicado en la carpeta del proyecto

In [20]:
import pandas as pd
pd.set_option('display.max_columns', None)
df = pd.read_csv("airline_passenger_satisfaction.csv")

## 2. Preprocesamiento

Para poder tratar los datos con nuestros modelos de aprendizaje debemos de normalizar los datos, tratar las variables categóricas y tratar las variables nulas (NaNs)

### 2.1 Selecció de característiques (Feature Selection)

In [28]:
# Muestra las primeras filas para ver la estructura
print(f"Dimensions del dataset: {df.shape}")
display(df.head())

Dimensions del dataset: (25976, 23)


,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,Female,Loyal Customer,52,Business travel,Eco,160,5,4,3,4,3,4,3,5,5,5,5,2,5,5,50,44.0,satisfied
1,Female,Loyal Customer,36,Business travel,Business,2863,1,1,3,1,5,4,5,4,4,4,4,3,4,5,0,0.0,satisfied
2,Male,disloyal Customer,20,Business travel,Eco,192,2,0,2,4,2,2,2,2,4,1,3,2,2,2,0,0.0,neutral or dissatisfied
3,Male,Loyal Customer,44,Business travel,Business,3377,0,0,0,2,3,4,4,1,1,1,1,3,1,4,0,6.0,satisfied
4,Female,Loyal Customer,49,Business travel,Eco,1182,2,3,4,3,4,1,2,2,2,2,2,4,2,4,0,20.0,satisfied


Nuestros datos contienen 25976 filas y 25 columnas de las cuales debemos eliminar las dos primera columnas `Unnamed` e `id` que no aportan información útil para la predicción.

In [29]:
# Eliminalos las columans que no son necesarias
cols_to_drop = ['Unnamed: 0', 'id']
df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

# Miramos las columnas restantes
print(df.columns.tolist())

['Gender', 'Customer Type', 'Age', 'Type of Travel', 'Class', 'Flight Distance', 'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Inflight service', 'Cleanliness', 'Departure Delay in Minutes', 'Arrival Delay in Minutes', 'satisfaction']


### 2.2 Limpieza y Transformación de Datos

#### Gestión de Valores Nulos (NaNs)

Los valores perdidos o nulos (NaNs) en un conjunto de datos pueden impedir el correcto funcionamiento de los algoritmos, ya que estos no pueden operar sobre datos inexistentes. Por ello, es imprescindible tratarlos antes de iniciar el entrenamiento.

Existen diversas técnicas para resolver este problema:
1.  **Eliminar filas:** Descartar los registros que contengan algún valor nulo.
2.  **Eliminar columnas:** Descartar la variable completa si tiene demasiados ausentes.
3.  **Imputación por valor fijo:** Sustituir el nulo por una constante.
4.  **Imputación estadística:** Sustituir el nulo por una medida de tendencia central, como la **media** o la mediana.

Nos hemos decantado por usar una estrategia híbrida:
* Para las **variables numéricas** (como *Arrival Delay*), sustituimos los valores faltantes con la **media** de la columna. Esto nos permite conservar la fila y no perder información valiosa del resto de características del pasajero.
* Para la **variable objetivo** (`satisfaction`), eliminaremos las filas que no tengan este dato. Al ser la etiqueta que queremos predecir (nuestra "verdad"), inventar este dato introduciría ruido y falsedades en el entrenamiento supervisado.

In [ ]:
# 1. Eliminamos las filas donde no tenemos la variable objetivo 'satisfaction'
df.dropna(subset=['satisfaction'], inplace=True)

# 2. Para el resto de columnas numéricas, rellenamos con la media (Imputación)
# Seleccionamos solo columnas numéricas para evitar errores con las de texto
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())

# Verificamos que ya no queden nulos
print(f"Nulos restantes: {df.isna().sum().sum()}")

Nulos restantes: 0
